In [1]:
import os 
import sys 
import json 
from pathlib import Path 

import torch 
import torch.nn as nn 
import numpy as np 
import matplotlib.pyplot as plt 
from torch.utils.data import DataLoader,TensorDataset 

project_root= Path.cwd()
while not(project_root /'requirements.txt').exists():
    if project_root == project_root.parent :
        raise RuntimeError("Could not find project root(looked for requirements.txt)")
    project_root= project_root.parent 
sys.path.insert(0,str(project_root))

from src.models.classification import ClassificationModelHandler 
from src.utils.training import test_model,training_loop 

In [2]:
device= torch.device('cuda'if torch.cuda.is_available()else 'cpu')
print(f"Using: {device}")

Using: cuda


In [ ]:
DATASET= "BreastMNIST"
FEATURE_GROUP= "concatenated"
FEATURE_VARIANT= "basic_resnet"

num_classes= 2 
batch_size= 32 
num_epochs= 500 
learning_rate= 0.001 

offline_features_dir= project_root /'experiments'/'models'/'fusion'/'offline'/DATASET /FEATURE_GROUP /FEATURE_VARIANT 
labels_dir= project_root /'experiments'/'labels'/DATASET 
save_dir= project_root /'experiments'/'models'/'classification'/'offline'/DATASET /FEATURE_VARIANT 
results_dir= project_root /'experiments'/'results'/'offline'/DATASET /FEATURE_VARIANT 

train_features_file= offline_features_dir /'train_features_concat.npy'
val_features_file= offline_features_dir /'val_features_concat.npy'
test_features_file= offline_features_dir /'test_features_concat.npy'

train_labels_file= labels_dir /'train_labels.npy'
val_labels_file= labels_dir /'val_labels.npy'
test_labels_file= labels_dir /'test_labels.npy'

model_name= f"classification_{DATASET.lower()}_{FEATURE_VARIANT}"

os.makedirs(save_dir,exist_ok= True)
os.makedirs(results_dir,exist_ok= True)

print(f"Dataset: {DATASET}")
print(f"Feature group: {FEATURE_GROUP}")
print(f"Feature variant: {FEATURE_VARIANT}")
print(f"Features directory: {offline_features_dir}")
print(f"Labels directory: {labels_dir}")
print(f"Model directory: {save_dir}")
print(f"Results directory: {results_dir}")

Dataset: BUSUCLM
Feature group: concatenated
Feature variant: basic_resnet
Features directory: /home/yrsobrinho/HybridFeatureFusionForBreastCancer/experiments/models/fusion/offline/BUSUCLM/concatenated/basic_resnet
Labels directory: /home/yrsobrinho/HybridFeatureFusionForBreastCancer/experiments/labels/BUSUCLM
Model directory: /home/yrsobrinho/HybridFeatureFusionForBreastCancer/experiments/models/classification/offline/BUSUCLM/basic_resnet
Results directory: /home/yrsobrinho/HybridFeatureFusionForBreastCancer/experiments/results/offline/BUSUCLM/basic_resnet


In [4]:
train_features= np.load(train_features_file)
val_features= np.load(val_features_file)
test_features= np.load(test_features_file)

train_labels= np.load(train_labels_file).squeeze().astype(np.int64)
val_labels= np.load(val_labels_file).squeeze().astype(np.int64)
test_labels= np.load(test_labels_file).squeeze().astype(np.int64)

print(f"Train features shape: {train_features.shape}")
print(f"Validation features shape: {val_features.shape}")
print(f"Test features shape: {test_features.shape}")
print(f"Train labels shape: {train_labels.shape}")
print(f"Validation labels shape: {val_labels.shape}")
print(f"Test labels shape: {test_labels.shape}")
print(f"Feature dimension: {train_features.shape [1]}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/yrsobrinho/HybridFeatureFusionForBreastCancer/experiments/models/fusion/offline/BUSUCLM/concatenated/basic_resnet/train_features_concat.npy'

In [ ]:
def build_loader(features,labels,batch_size,shuffle= False):
    features_tensor= torch.FloatTensor(features)
    labels_tensor= torch.LongTensor(labels)
    dataset= TensorDataset(features_tensor,labels_tensor)
    return DataLoader(dataset,batch_size= batch_size,shuffle= shuffle)

val_loader= build_loader(val_features,val_labels,batch_size= batch_size,shuffle= False)
test_loader= build_loader(test_features,test_labels,batch_size= batch_size,shuffle= False)

print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of test batches: {len(test_loader)}")

In [ ]:
train_loader= build_loader(train_features,train_labels,batch_size= batch_size,shuffle= True)

print(f"Number of training batches: {len(train_loader)}")

In [ ]:
sequential_hyperparams= {
'input_dim':train_features.shape [1 ],
'learning_rate':learning_rate 
}

classifier= ClassificationModelHandler(
device= device,
num_classes= num_classes,
sequential_hyperparams= sequential_hyperparams 
)

print("Model:")
print(classifier)

In [ ]:
criterion= nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(classifier.model.parameters(),lr= learning_rate)

history= training_loop(
model= classifier.model,
train_loader= train_loader,
val_loader= val_loader,
criterion= criterion,
optimizer= optimizer,
device= device,
num_epochs= num_epochs,
save_dir= str(save_dir),
model_name= model_name,
extract_features= False,
save_best_only= True,
save_all_epochs= False 
)

best_model_path= save_dir /f"best_{model_name}.pth"

print("\nTraining completed.")
print(f"Best model path: {best_model_path}")

In [ ]:
best_model_path= save_dir /f"best_{model_name}.pth"

if best_model_path.exists():
    classifier.model.load_state_dict(torch.load(best_model_path,map_location= device))
    print(f"Best model loaded from: {best_model_path}")
else :
    print("Warning: best model file not found. Using current model weights.")

classifier.model.eval()

In [ ]:
criterion= nn.CrossEntropyLoss()

hyperparams= {
'dataset':DATASET,
'feature_group':FEATURE_GROUP,
'feature_variant':FEATURE_VARIANT,
'num_classes':num_classes,
'batch_size':batch_size,
'num_epochs':num_epochs,
'learning_rate':learning_rate,
'input_dim':train_features.shape [1 ]
}

output_file= results_dir /'test_results.txt'
metrics_file= results_dir /'metrics_summary.json'

test_loss,accuracy,precision,recall,f1,auc= test_model(
model= classifier.model,
dataloader= test_loader,
criterion= criterion,
device= device,
hyperparams= hyperparams,
output_file= str(output_file),
extract_features= False,
compute_detailed_metrics= True 
)

summary= {
'dataset':DATASET,
'feature_group':FEATURE_GROUP,
'feature_variant':FEATURE_VARIANT,
'model_name':model_name,
'best_model_path':str(best_model_path),
'results_file':str(output_file),
'metrics':{
'test_loss':float(test_loss),
'accuracy':float(accuracy),
'precision':float(precision),
'recall':float(recall),
'f1_score':float(f1),
'auc':float(auc)
}
}

with open(metrics_file,'w')as f :
    json.dump(summary,f,indent= 4)

print(f"\n{'='*50}")
print("FINAL TEST RESULTS:")
print(f"{'='*50}")
print(f"Test Loss: {test_loss :.4f}")
print(f"Accuracy: {accuracy :.4f}")
print(f"Precision: {precision :.4f}")
print(f"Recall: {recall :.4f}")
print(f"F1-Score: {f1 :.4f}")
print(f"AUC: {auc :.4f}")
print(f"{'='*50}")
print(f"Detailed results saved at: {output_file}")
print(f"Metrics summary saved at: {metrics_file}")

## Multi-run evaluation

Trains the model above **N_RUNS** times from scratch (seeds `0..N_RUNS-1`) and reports **mean ± std** plus a **95% confidence interval** (Student's t) for every test metric, together with per-run and aggregated confusion matrices.

All artefacts are written under `reports/offline_classify/` (see `reports/README.md`). Adjust `N_RUNS` below if this is too slow — quantum runs in particular are expensive.

In [ ]:
# ==========================================================================
# MULTI-RUN EVALUATION  (N_RUNS independent runs, seeds 0..N_RUNS-1)
# Retrains the model from scratch each run, then reports mean +/- std and a
# 95% CI (Student's t) for every test metric, plus per-run and aggregated
# confusion matrices. Everything is written under `reports/offline_classify/`.
# Run the setup cells above first (imports, data loaders, config, model class).
# ==========================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.stats import t as t_dist
from sklearn.metrics import confusion_matrix

N_RUNS = 5
CONFIDENCE_LEVEL = 0.95
NUM_CLASSES = num_classes
EPOCHS_PER_RUN = num_epochs
RUN_TAG = model_name
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]

# Output location (rename the subfolder freely; just update the string below).
try:
    _root = str(project_root)
except NameError:
    _root = os.getcwd()
REPORTS_DIR = os.path.join(_root, "reports", "offline_classify")
multi_run_dir = os.path.join(REPORTS_DIR, "multi_run", "raw")
os.makedirs(multi_run_dir, exist_ok=True)


def _build_run_model(seed):
    """Return (model, optimizer, model_name) freshly built for this run."""
    handler = ClassificationModelHandler(
        device=device, num_classes=num_classes,
        sequential_hyperparams=sequential_hyperparams,
    )
    m = handler.model
    opt = torch.optim.Adam(m.parameters(), lr=learning_rate)
    name = f"{model_name}_run{seed + 1}"
    return m, opt, name

def _get_predictions(model, dataloader, device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            y_pred.extend(outputs.argmax(dim=1).cpu().numpy().tolist())
            y_true.extend(np.asarray(labels).squeeze().reshape(-1).tolist())
    return np.array(y_true), np.array(y_pred)


def _plot_confusion_matrix(cm, save_path, title, class_names=None, normalize=False):
    cm = np.asarray(cm, dtype=float)
    if normalize:
        row_sums = cm.sum(axis=1, keepdims=True)
        disp = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
        cell_fmt = lambda v: f"{v:.1%}"
    else:
        disp = cm
        cell_fmt = lambda v: f"{int(round(v))}"
    n = cm.shape[0]
    names = class_names if class_names is not None else [str(i) for i in range(n)]
    fig, ax = plt.subplots(figsize=(1.4 * n + 2.5, 1.4 * n + 2.5))
    im = ax.imshow(disp, cmap="Blues", vmin=0)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticklabels(names)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    thresh = disp.max() / 2 if disp.max() > 0 else 0.5
    for i in range(n):
        for j in range(n):
            ax.text(j, i, cell_fmt(disp[i, j]), ha="center", va="center",
                    color="white" if disp[i, j] > thresh else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


run_results = []
run_confusion_matrices = []

for run_idx in range(N_RUNS):
    seed = run_idx
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    print(f"\n{'=' * 80}")
    print(f"RUN {run_idx + 1}/{N_RUNS} (seed={seed}) - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'=' * 80}\n")

    run_model, run_optimizer, run_model_name = _build_run_model(seed)
    run_criterion = nn.CrossEntropyLoss()

    best_path = os.path.join(multi_run_dir, f"best_{run_model_name}.pth")
    if os.path.isfile(best_path):
        os.remove(best_path)

    training_loop(
        model=run_model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=run_criterion,
        optimizer=run_optimizer,
        device=device,
        num_epochs=EPOCHS_PER_RUN,
        save_dir=multi_run_dir,
        model_name=run_model_name,
        extract_features=False,
        save_best_only=True,
        save_all_epochs=False,
    )

    if os.path.isfile(best_path):
        run_model.load_state_dict(torch.load(best_path, map_location=device))
    else:
        print(f"⚠ Warning: run {run_idx + 1} never beat the initial val_acc baseline — "
              f"testing with final-epoch weights instead of a best checkpoint.")
    run_model.eval()

    test_loss, test_acc, test_prec, test_rec, test_f1, test_auc = test_model(
        model=run_model,
        dataloader=test_loader,
        criterion=run_criterion,
        device=device,
        hyperparams={"dataset": DATASET, "feature_group": FEATURE_GROUP,
                    "feature_variant": FEATURE_VARIANT, "num_classes": num_classes,
                    "batch_size": batch_size, "num_epochs": num_epochs,
                    "learning_rate": learning_rate, "input_dim": train_features.shape[1]},
        output_file=os.path.join(multi_run_dir, f"test_log_run{run_idx + 1}.txt"),
        extract_features=False,
        compute_detailed_metrics=True,
    )

    run_labels, run_preds = _get_predictions(run_model, test_loader, device)
    run_cm = confusion_matrix(run_labels, run_preds, labels=list(range(NUM_CLASSES)))
    run_confusion_matrices.append(run_cm)
    _plot_confusion_matrix(
        run_cm,
        save_path=os.path.join(multi_run_dir, f"confusion_matrix_run{run_idx + 1}.png"),
        title=f"Confusion Matrix - Run {run_idx + 1}",
        class_names=CLASS_NAMES,
    )

    run_record = {
        "run": run_idx + 1,
        "loss": test_loss,
        "accuracy": test_acc,
        "precision": test_prec,
        "recall": test_rec,
        "f1": test_f1,
        "auc": test_auc,
    }
    _gamma = getattr(run_model, "gamma", None)
    if _gamma is not None:
        run_record["gamma"] = float(_gamma.item()) if hasattr(_gamma, "item") else float(_gamma)
    run_results.append(run_record)

    print(f"\n✓ Run {run_idx + 1} test accuracy: {test_acc:.4f}")

print(f"\n{'=' * 80}")
print(f"All {N_RUNS} runs complete")
print(f"{'=' * 80}")

metrics_keys = ["loss", "accuracy", "precision", "recall", "f1", "auc"]
if all("gamma" in r for r in run_results):
    metrics_keys.append("gamma")

stats = {}
for key in metrics_keys:
    values = np.array([r[key] for r in run_results], dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if n > 1 else 0.0
    if n > 1:
        sem = std / np.sqrt(n)
        t_crit = float(t_dist.ppf((1 + CONFIDENCE_LEVEL) / 2, df=n - 1))
        margin = t_crit * sem
    else:
        margin = 0.0
    stats[key] = {
        "mean": mean, "std": std,
        "min": float(np.min(values)), "max": float(np.max(values)),
        "ci_lower": mean - margin, "ci_upper": mean + margin,
    }

cm_sum = np.stack(run_confusion_matrices).sum(axis=0)
cm_sum_path = os.path.join(multi_run_dir, "confusion_matrix_sum.png")
_plot_confusion_matrix(cm_sum.astype(int), cm_sum_path,
                       f"Confusion Matrix - Sum over {N_RUNS} runs", CLASS_NAMES)
cm_mean_path = os.path.join(multi_run_dir, "confusion_matrix_mean_normalized.png")
_plot_confusion_matrix(cm_sum, cm_mean_path,
                       f"Confusion Matrix - Row-Normalized (mean over {N_RUNS} runs)",
                       CLASS_NAMES, normalize=True)

stats_path = os.path.join(REPORTS_DIR, f"multi_run_statistics_{RUN_TAG}.txt")
report_meta = {
    "Dataset": DATASET,
    "Feature Group": FEATURE_GROUP,
    "Feature Variant": FEATURE_VARIANT,
    "Epochs per run": num_epochs,
    "Batch Size": batch_size,
    "Learning Rate": learning_rate,
    "Input Dim": train_features.shape[1],
}
with open(stats_path, "w", encoding="utf-8") as f:
    f.write("=" * 80 + "\n")
    f.write(f"MULTI-RUN STATISTICS ({N_RUNS} runs)\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    for _k, _v in report_meta.items():
        f.write(f"{_k}: {_v}\n")
    f.write("\n")

    f.write("PER-RUN TEST RESULTS\n")
    f.write("-" * 80 + "\n")
    header = f"{'Run':<5}{'Loss':<10}{'Accuracy':<10}{'Precision':<12}{'Recall':<10}{'F1':<10}{'AUC':<10}"
    if "gamma" in metrics_keys:
        header += f"{'Gamma':<10}"
    f.write(header + "\n")
    for r in run_results:
        line = (f"{r['run']:<5}{r['loss']:<10.4f}{r['accuracy']:<10.4f}"
                f"{r['precision']:<12.4f}{r['recall']:<10.4f}{r['f1']:<10.4f}{r['auc']:<10.4f}")
        if "gamma" in metrics_keys:
            line += f"{r['gamma']:<10.4f}"
        f.write(line + "\n")
    f.write("\n")

    f.write(f"SUMMARY STATISTICS (mean +/- std, min, max, {int(CONFIDENCE_LEVEL * 100)}% CI [t-distribution])\n")
    f.write("-" * 80 + "\n")
    for key in metrics_keys:
        s = stats[key]
        f.write(f"{key.capitalize():<12}: {s['mean']:.4f} +/- {s['std']:.4f}"
                f"  (min: {s['min']:.4f}, max: {s['max']:.4f})"
                f"  {int(CONFIDENCE_LEVEL * 100)}% CI: [{s['ci_lower']:.4f}, {s['ci_upper']:.4f}]\n")
    f.write("\n")

    labels_for_header = CLASS_NAMES if CLASS_NAMES is not None else [str(i) for i in range(cm_sum.shape[0])]
    f.write(f"CONFUSION MATRIX - SUMMED OVER {N_RUNS} RUNS (rows = true, cols = predicted)\n")
    f.write("-" * 80 + "\n")
    f.write("            " + "".join(f"{lbl:>12}" for lbl in labels_for_header) + "\n")
    for lbl, row in zip(labels_for_header, cm_sum.astype(int)):
        f.write(f"{lbl:>12}" + "".join(f"{v:>12d}" for v in row) + "\n")
    f.write(f"\nSaved plot: {cm_sum_path}\n\n")

    f.write(f"CONFUSION MATRIX - ROW-NORMALIZED MEAN OVER {N_RUNS} RUNS\n")
    f.write("-" * 80 + "\n")
    cm_norm = cm_sum.astype(float) / cm_sum.sum(axis=1, keepdims=True)
    f.write("            " + "".join(f"{lbl:>12}" for lbl in labels_for_header) + "\n")
    for lbl, row in zip(labels_for_header, cm_norm):
        f.write(f"{lbl:>12}" + "".join(f"{v:>12.2%}" for v in row) + "\n")
    f.write(f"\nSaved plot: {cm_mean_path}\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("END OF REPORT\n")
    f.write("=" * 80 + "\n")

print(f"✓ Multi-run statistics saved to: {stats_path}")
print()
for key in metrics_keys:
    s = stats[key]
    print(f"  {key.capitalize():<12}: {s['mean']:.4f} +/- {s['std']:.4f}"
          f"  (min: {s['min']:.4f}, max: {s['max']:.4f})"
          f"  {int(CONFIDENCE_LEVEL * 100)}% CI: [{s['ci_lower']:.4f}, {s['ci_upper']:.4f}]")
print("\nSummed confusion matrix:")
print(cm_sum.astype(int))
